# WanGP no Google Colab — um play e pronto

Gera **vídeo e imagem** por inteligência artificial (Wan 2.1, Wan 2.2, LTX-Video) numa
página com botões — você não escreve código nenhum.

### Antes de apertar o play

Menu do topo → **Ambiente de execução** → **Alterar o tipo de ambiente de execução** →
marque **T4 GPU** → **Salvar**.

Sem isso nada funciona. A célula abaixo trava de propósito e avisa, se faltar.

### Depois

Aperte o play na célula, autorize o Google Drive quando ele pedir, e **espere o link azul
terminado em `.gradio.live`**. Esse link é o painel. Na primeira vez demora, porque ele
está baixando o programa e o modelo.

**Não feche esta aba** enquanto estiver usando o painel.


In [ ]:
#@title ## WanGP no Colab — aperte o play e espere o link azul { display-mode: "form" }
#@markdown Nao precisa mexer em nada. Se quiser guardar os videos no seu Google Drive,
#@markdown deixe a caixa marcada (ele vai pedir sua autorizacao no meio do caminho).
USAR_GOOGLE_DRIVE = True  #@param {type:"boolean"}
#@markdown Pasta dentro do seu Drive onde ficam os modelos, as LoRAs e os videos:
PASTA_NO_DRIVE = "WanGP"  #@param {type:"string"}

import os, sys, shutil, subprocess

# ------------------------------------------------------------------ 1. a placa
gpu = ""
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True).stdout.strip()
except Exception:
    pass

if not gpu:
    print()
    print("!" * 76)
    print("!!  PARE. ESTA SESSAO ESTA SEM PLACA DE VIDEO - E SEM ELA O PAINEL NAO LIGA.")
    print("!!")
    print("!!  Sao DUAS causas possiveis. Veja qual foi a sua:")
    print("!!")
    print("!!  1) VOCE AINDA NAO ESCOLHEU A PLACA.")
    print("!!     Menu do topo  ->  Ambiente de execucao  ->  Alterar o tipo de")
    print("!!     ambiente de execucao  ->  marque  T4 GPU  ->  Salvar.")
    print("!!     Depois aperte o play aqui de novo.")
    print("!!")
    print("!!  2) O COLAB AVISOU  -nao e possivel conectar a GPU / limites de uso-")
    print("!!     e voce clicou em  -Conectar sem GPU-.")
    print("!!     Entao a sua cota gratuita de placa de video acabou por hoje.")
    print("!!     Nao ha nada para ajustar aqui: quem fechou a torneira foi o Google.")
    print("!!     Suas saidas sao duas:")
    print("!!       . esperar (costuma voltar em algumas horas, ou no dia seguinte)")
    print("!!       . abrir este mesmo link com a OUTRA conta do Google")
    print("!" * 76)
    print()
    raise RuntimeError("Sem placa de video. Leia o aviso acima: ou falta escolher T4 GPU, ou a cota diaria de GPU do Colab acabou por hoje.")

print("Placa de video: " + gpu)

# ------------------------------------------------------------------ 2. o Drive
PASTA = None
if USAR_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    PASTA = '/content/drive/MyDrive/' + PASTA_NO_DRIVE.strip('/ ')
    for sub in ('hf-cache', 'loras', 'outputs'):
        os.makedirs(os.path.join(PASTA, sub), exist_ok=True)
    os.environ['HF_HOME'] = os.path.join(PASTA, 'hf-cache')
    os.environ['HUGGINGFACE_HUB_CACHE'] = os.path.join(PASTA, 'hf-cache')
    print("Drive conectado. Tudo sera salvo em: " + PASTA)
else:
    print("Sem Drive: os videos somem quando o Colab fechar. Baixe antes de sair.")

os.environ['TRITON_CACHE_DIR'] = '/tmp/triton_cache'

# ------------------------------------------------------------------ 3. o programa
%cd /content
if not os.path.isdir('/content/Wan2GP'):
    print("\nBaixando o programa (uma vez so)...")
    !git clone -q https://github.com/deepbeepmeep/Wan2GP.git
%cd /content/Wan2GP

import torch
TORCH_BASE = torch.__version__.split('+')[0]
CUDA_TAG = ('cu' + torch.version.cuda.replace('.', '')) if torch.version.cuda else 'cu126'

print("\nInstalando as pecas que faltam. Demora de 5 a 12 minutos - pode deixar rodando.")
!pip install -q -r requirements.txt

def estado_torch():
    r = subprocess.run([sys.executable, "-c",
        "import torch;print(torch.__version__);print(torch.cuda.is_available())"],
        capture_output=True, text=True).stdout.split()
    return (r[0], r[1]) if len(r) > 1 else ("?", "?")

ver, usa = estado_torch()
if usa != "True":
    print("A instalacao trocou o motor de calculo por uma versao sem placa. Consertando...")
    !pip install -q --force-reinstall torch=={TORCH_BASE} --index-url https://download.pytorch.org/whl/{CUDA_TAG}
    ver, usa = estado_torch()
print("Motor de calculo: torch " + ver + " | usa a placa: " + usa)

# ------------------------------------------------------------------ 4. atalhos para o Drive
if PASTA:
    for nome in ('loras', 'outputs'):
        alvo = '/content/Wan2GP/' + nome
        if os.path.islink(alvo):
            os.unlink(alvo)
        elif os.path.isdir(alvo):
            shutil.rmtree(alvo, ignore_errors=True)
        os.symlink(os.path.join(PASTA, nome), alvo)
    os.makedirs(os.path.join(PASTA, 'loras', 'wan'), exist_ok=True)

# ------------------------------------------------------------------ 5. ligar
print("\n" + "=" * 72)
print("LIGANDO O PAINEL. Na primeira vez ele ainda baixa o modelo (varios GB),")
print("entao o link pode levar de 5 a 15 minutos para aparecer. E normal.")
print("Quando aparecer, clique no endereco que termina em  .gradio.live")
print("=" * 72 + "\n")

!python wgp.py --share --listen --server-port 7860 --attention sdpa --profile 4

